# Calculus, noise and uncertainty · Modules 5–6
Use Python 3 and NumPy. Name the assumption behind each estimate. A numerical derivative checks a computation locally; an uncertainty estimate applies to its declared sampling model.


In [1]:
import numpy as np
np.set_printoptions(precision=8,suppress=True)
rng=np.random.default_rng(206)
def loss(theta):
    x,y=theta
    return x*x+3*x*y+y*y
def analytic(theta):
    x,y=theta
    return np.array([2*x+3*y,3*x+2*y])
def central_gradient(fn,theta,h):
    result=np.zeros_like(theta,dtype=float)
    for j in range(len(theta)):
        plus=theta.copy();minus=theta.copy()
        plus[j]+=h;minus[j]-=h
        result[j]=(fn(plus)-fn(minus))/(2*h)
    return result
theta=np.array([1.,2.]);g=analytic(theta)
print('analytic gradient:',g)
for h in [1e-2,1e-4,1e-6,1e-8]:
    numeric=central_gradient(loss,theta,h)
    print('h, numerical, max abs error:',h,numeric,np.max(np.abs(numeric-g)))
assert np.allclose(central_gradient(loss,theta,1e-4),g,atol=1e-8)
# A nonquadratic derivative shows truncation error as well.
for h in [.1,.01,1e-5,1e-8]:
    numeric=((2+h)**3-(2-h)**3)/(2*h)
    print('cubic step, error:',h,numeric-12)


analytic gradient: [8. 7.]
h, numerical, max abs error: 0.01 [8. 7.] 9.592326932761353e-14
h, numerical, max abs error: 0.0001 [8. 7.] 1.921307557495311e-11
h, numerical, max abs error: 1e-06 [8. 7.] 6.581331035704352e-10
h, numerical, max abs error: 1e-08 [7.99999995 6.99999987] 1.3136013876646757e-07
cubic step, error: 0.1 0.010000000000008669
cubic step, error: 0.01 9.999999984700025e-05
cubic step, error: 1e-05 2.1183943488267687e-10
cubic step, error: 1e-08 -1.1733857263607206e-07


## Chain-rule check
Write a=w*x+b and L=(a-target)^2/2. Derive the parameter gradients before running the cell. Do not modify parameters while computing a backward pass.


In [2]:
x=2.;w=3.;b=1.;target=5.
a=w*x+b;r=a-target
print('forward:',a,'loss:',r*r/2)
print('dL/dw, dL/db, dL/dx:',r*x,r,r*w)
assert np.allclose([r*x,r,r*w],[4,2,6])
prior=.2;recall=.8;false_rate=.1
tp=prior*recall;fp=(1-prior)*false_rate
posterior=tp/(tp+fp)
assert np.isclose(posterior,2/3)
print('P(object | flag):',posterior)
print('uniform [0,2], P([.5,1.5]):',(1.5-.5)/2)


forward: 7.0 loss: 2.0
dL/dw, dL/db, dL/dx: 4.0 2.0 6.0
P(object | flag): 0.6666666666666666
uniform [0,2], P([.5,1.5]): 0.5


## Gaussian noise and averages
These are unclipped floating-point measurements. Clipping would change the model. Estimate variance reduction over many independent trials and compare it with sigma^2/N.


In [3]:
sigma=2.;n=16;trials=20000
measurements=50.+rng.normal(0,sigma,(trials,n))
averages=measurements.mean(axis=1)
estimated=averages.var(ddof=1)
expected=sigma**2/n
print('mean estimate:',averages.mean(),'variance of mean:',estimated,'model variance:',expected)
assert abs(estimated-expected)<.03
X=np.array([-1.,0.,1.]);Y=X**2
covariance=np.mean((X-X.mean())*(Y-Y.mean()))
assert np.isclose(covariance,0)
print('dependent X,Y with covariance zero:',covariance)


mean estimate: 49.998626719874004 variance of mean: 0.25233604477025545 model variance: 0.25
dependent X,Y with covariance zero: 0.0


## Accuracy sampling uncertainty
The independent Bernoulli model is explicit. Bootstrap the observed outcomes with replacement. This simple percentile interval is approximate and cannot create unseen failure modes.


In [4]:
correct=rng.binomial(1,.8,100)
accuracy=correct.mean();se=np.sqrt(accuracy*(1-accuracy)/len(correct))
replicates=correct[rng.integers(0,len(correct),(4000,len(correct)))].mean(axis=1)
interval=np.quantile(replicates,[.025,.975])
print('accuracy:',accuracy,'N:',len(correct),'plug-in SE:',se)
print('rough normal interval:',accuracy+np.array([-1,1])*1.96*se)
print('bootstrap percentile interval:',interval)
assert 0<=interval[0]<=interval[1]<=1
print('3 successes out of 3: naive plug-in SE is',np.sqrt(1*(1-1)/3),'and is misleading.')


accuracy: 0.81 N: 100 plug-in SE: 0.03923009049186606
rough normal interval: [0.73310902 0.88689098]
bootstrap percentile interval: [0.73 0.88]
3 successes out of 3: naive plug-in SE is 0.0 and is misleading.


## Your experiment
Sweep finite-difference step size on a smooth nonquadratic loss. Predict when error falls and when roundoff matters. Then vary independent sample count and plot or tabulate variance of the mean. Finally, explain why correlated video frames need a group-based sampling model rather than treating every frame as independent.
